In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 01 - Exploratory Data Analysis (EDA)\n",
    "## Smart Product Pricing - Initial Data Exploration\n",
    "\n",
    "**Objectives:**\n",
    "1. Load and inspect train/test data\n",
    "2. Analyze target distribution (price)\n",
    "3. Explore text features (catalog_content)\n",
    "4. Check image availability\n",
    "5. Identify data quality issues\n",
    "6. Generate initial insights for feature engineering"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Imports\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "\n",
    "print(\"✓ Libraries imported\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load datasets\n",
    "df_train = pd.read_csv('../data/train.csv')\n",
    "df_test = pd.read_csv('../data/test.csv')\n",
    "\n",
    "print(f\"Train shape: {df_train.shape}\")\n",
    "print(f\"Test shape: {df_test.shape}\")\n",
    "print(f\"\\nTrain columns: {df_train.columns.tolist()}\")\n",
    "print(f\"Test columns: {df_test.columns.tolist()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# First look at train data\n",
    "df_train.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Data info\n",
    "df_train.info()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for missing values\n",
    "missing = df_train.isnull().sum()\n",
    "missing_pct = (missing / len(df_train)) * 100\n",
    "\n",
    "missing_df = pd.DataFrame({\n",
    "    'Missing': missing,\n",
    "    'Percentage': missing_pct\n",
    "}).sort_values('Missing', ascending=False)\n",
    "\n",
    "print(\"Missing Values:\")\n",
    "print(missing_df[missing_df['Missing'] > 0])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Target Analysis (Price)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Price statistics\n",
    "print(\"Price Statistics:\")\n",
    "print(df_train['price'].describe())\n",
    "print(f\"\\nSkewness: {df_train['price'].skew():.2f}\")\n",
    "print(f\"Kurtosis: {df_train['price'].kurtosis():.2f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Price distribution\n",
    "fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n",
    "\n",
    "# Raw price\n",
    "axes[0].hist(df_train['price'], bins=50, edgecolor='black')\n",
    "axes[0].set_title('Price Distribution (Raw)')\n",
    "axes[0].set_xlabel('Price')\n",
    "axes[0].set_ylabel('Frequency')\n",
    "\n",
    "# Log price\n",
    "axes[1].hist(np.log1p(df_train['price']), bins=50, edgecolor='black', color='green')\n",
    "axes[1].set_title('Price Distribution (Log1p)')\n",
    "axes[1].set_xlabel('Log1p(Price)')\n",
    "axes[1].set_ylabel('Frequency')\n",
    "\n",
    "# Box plot\n",
    "axes[2].boxplot(df_train['price'])\n",
    "axes[2].set_title('Price Box Plot')\n",
    "axes[2].set_ylabel('Price')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"✓ Log transformation stabilizes the distribution\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Outlier detection\n",
    "Q1 = df_train['price'].quantile(0.25)\n",
    "Q3 = df_train['price'].quantile(0.75)\n",
    "IQR = Q3 - Q1\n",
    "\n",
    "lower_bound = Q1 - 1.5 * IQR\n",
    "upper_bound = Q3 + 1.5 * IQR\n",
    "\n",
    "outliers = df_train[(df_train['price'] < lower_bound) | (df_train['price'] > upper_bound)]\n",
    "\n",
    "print(f\"Outliers (IQR method): {len(outliers)} ({len(outliers)/len(df_train)*100:.2f}%)\")\n",
    "print(f\"Price range: [{df_train['price'].min():.2f}, {df_train['price'].max():.2f}]\")\n",
    "print(f\"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Text Analysis (catalog_content)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Text length statistics\n",
    "df_train['text_len'] = df_train['catalog_content'].fillna('').str.len()\n",
    "df_train['text_words'] = df_train['catalog_content'].fillna('').str.split().str.len()\n",
    "\n",
    "print(\"Text Length Statistics:\")\n",
    "print(df_train[['text_len', 'text_words']].describe())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Text length distribution\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "axes[0].hist(df_train['text_len'], bins=50, edgecolor='black')\n",
    "axes[0].set_title('Character Length Distribution')\n",
    "axes[0].set_xlabel('Characters')\n",
    "axes[0].set_ylabel('Frequency')\n",
    "\n",
    "axes[1].hist(df_train['text_words'], bins=50, edgecolor='black', color='orange')\n",
    "axes[1].set_title('Word Count Distribution')\n",
    "axes[1].set_xlabel('Words')\n",
    "axes[1].set_ylabel('Frequency')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Sample text examples\n",
    "print(\"Sample Product Descriptions:\\n\")\n",
    "for i, text in enumerate(df_train['catalog_content'].head(5)):\n",
    "    print(f\"{i+1}. {text[:200]}...\\n\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Common patterns detection\n",
    "import re\n",
    "\n",
    "patterns = {\n",
    "    'has_quantity': r'\\d+\\s*x\\s*\\d+',\n",
    "    'has_pack': r'pack\\s*of\\s*\\d+',\n",
    "    'has_weight_kg': r'\\d+\\.?\\d*\\s*kg',\n",
    "    'has_weight_g': r'\\d+\\.?\\d*\\s*g\\b',\n",
    "    'has_volume_ml': r'\\d+\\.?\\d*\\s*ml',\n",
    "    'has_volume_l': r'\\d+\\.?\\d*\\s*l\\b',\n",
    "}\n",
    "\n",
    "for name, pattern in patterns.items():\n",
    "    df_train[name] = df_train['catalog_content'].fillna('').str.contains(pattern, case=False, regex=True)\n",
    "    count = df_train[name].sum()\n",
    "    pct = count / len(df_train) * 100\n",
    "    print(f\"{name:20s}: {count:6d} ({pct:5.2f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Image Availability"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check image links\n",
    "if 'image_link' in df_train.columns:\n",
    "    print(f\"Images available: {df_train['image_link'].notna().sum()} / {len(df_train)}\")\n",
    "    print(f\"Missing images: {df_train['image_link'].isna().sum()}\")\n",
    "    \n",
    "    # Check downloaded images\n",
    "    images_dir = Path('../data/images')\n",
    "    if images_dir.exists():\n",
    "        downloaded = sum(1 for sid in df_train['sample_id'] if (images_dir / f\"{sid}.jpg\").exists())\n",
    "        print(f\"Downloaded images: {downloaded} / {len(df_train)}\")\n",
    "else:\n",
    "    print(\"No image_link column found\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Price vs Features"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Price vs text length\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "axes[0].scatter(df_train['text_len'], df_train['price'], alpha=0.3)\n",
    "axes[0].set_xlabel('Text Length (characters)')\n",
    "axes[0].set_ylabel('Price')\n",
    "axes[0].set_title('Price vs Text Length')\n",
    "\n",
    "axes[1].scatter(df_train['text_words'], df_train['price'], alpha=0.3, color='orange')\n",
    "axes[1].set_xlabel('Word Count')\n",
    "axes[1].set_ylabel('Price')\n",
    "axes[1].set_title('Price vs Word Count')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(f\"Correlation (text_len vs price): {df_train[['text_len', 'price']].corr().iloc[0,1]:.3f}\")\n",
    "print(f\"Correlation (text_words vs price): {df_train[['text_words', 'price']].corr().iloc[0,1]:.3f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Price by pattern presence\n",
    "pattern_cols = [col for col in df_train.columns if col.startswith('has_')]\n",
    "\n",
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, col in enumerate(pattern_cols):\n",
    "    data = [df_train[df_train[col]]['price'], df_train[~df_train[col]]['price']]\n",
    "    axes[i].boxplot(data, labels=['Present', 'Absent'])\n",
    "    axes[i].set_title(f'Price Distribution: {col}')\n",
    "    axes[i].set_ylabel('Price')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Key Insights\n",
    "\n",
    "### Target (Price):\n",
    "- Heavy right-skewed distribution → Use log1p transformation\n",
    "- Wide range with some outliers\n",
    "- Log transformation stabilizes variance\n",
    "\n",
    "### Text Features:\n",
    "- Variable length descriptions\n",
    "- Common patterns: quantities, packs, weights, volumes\n",
    "- Weak correlation with raw text length\n",
    "\n",
    "### Images:\n",
    "- Check availability rate\n",
    "- Handle missing images gracefully\n",
    "\n",
    "### Next Steps:\n",
    "1. Parse quantities and units systematically\n",
    "2. Extract brand information\n",
    "3. Create TF-IDF features from text\n",
    "4. Extract CLIP embeddings from images\n",
    "5. Build ensemble model"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}